In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import os
from pyspark.sql import SparkSession

spark_master = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")

spark = (
    SparkSession.builder
    .appName("T032-LinearRegression")
    .master(spark_master)
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print("SPARK_MASTER:", spark_master)
print("spark.sparkContext.master:", spark.sparkContext.master)
print("Motor do Spark versão:", spark.version)

SPARK_MASTER: spark://spark-master:7077
spark.sparkContext.master: spark://spark-master:7077
Motor do Spark versão: 3.2.1


In [2]:
#Carregar os Dados
caminho_dados = "/dataset/Indian_Weather_Dataset.parquet"
# Carrega o dataset
df = spark.read.parquet(caminho_dados)
print(f"Total de registros: {df.count()}")

# Divide os dados em Treino e Teste
df_treino, df_teste = df.randomSplit([0.7, 0.3], seed=42)
print(f"Registros de Treino: {df_treino.count()}")
print(f"Registros de Teste: {df_teste.count()}")

Total de registros: 46082160
Registros de Treino: 32259408
Registros de Teste: 13822752


In [3]:
colunas_features = [
    "humidity_pct", "pressure_hPa", "dew_point_C", 
    "solar_radiation_Wm2", "cloud_cover_pct", "wind_speed_ms", 
    "wind_dir_sin", "wind_dir_cos", "cape", "et0_mm", "precip_mm"
]

#Agrupar as features
assembler = VectorAssembler(inputCols=colunas_features, outputCol="raw_features")

#StandardScaller
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", 
                      withStd=True, withMean=True)

#Linear_regression
lr = LinearRegression(featuresCol="scaled_features", labelCol="temperature_C", 
                      regParam=0.1, elasticNetParam=0.0, solver="auto")

#Pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])
print("Pipeline configurada com sucesso")

Pipeline configurada com sucesso


In [4]:
print("Treino")

# Treina o modelo
modelo_treinado = pipeline.fit(df_treino)

# Aplica o modelo nos dados
previsoes = modelo_treinado.transform(df_teste)
evaluator_rmse = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="temperature_C", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(previsoes)
mae = evaluator_mae.evaluate(previsoes)
r2 = evaluator_r2.evaluate(previsoes)

print(f"\n--- RESULTADOS DO MODELO ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

Treino

--- RESULTADOS DO MODELO ---
RMSE: 1.6435
MAE:  1.1371
R2:   0.9524


In [5]:
# Extrai o modelo do Pipeline
lr_model = modelo_treinado.stages[-1]

print("Intercepto (Temperatura Base):", round(lr_model.intercept, 4))
print("\nCoeficientes gerados pela Regularização Ridge:")

for feature, coef in zip(colunas_features, lr_model.coefficients):
    print(f"- {feature}: {round(coef, 4)}")

Intercepto (Temperatura Base): 23.6115

Coeficientes gerados pela Regularização Ridge:
- humidity_pct: -5.6658
- pressure_hPa: 0.3989
- dew_point_C: 6.7973
- solar_radiation_Wm2: -2.798
- cloud_cover_pct: 0.2588
- wind_speed_ms: -0.2243
- wind_dir_sin: -0.1112
- wind_dir_cos: -0.0769
- cape: 0.0
- et0_mm: 3.605
- precip_mm: 0.1435


In [ ]:
spark.stop()
print("Spark finalizado.")